# 01 — Target preparation (reproducible)

Downloads PDB 4ZQK, verifies the chain mapping from the RCSB Data API, and computes a PD-L1/PD-1 heavy-atom contact map.


In [ ]:
import csv, hashlib, math, pathlib, requests
PDB_ID = '4ZQK'
OUT = pathlib.Path('/content/pdl1_target')
OUT.mkdir(exist_ok=True)
pdb_bytes = requests.get(f'https://files.rcsb.org/download/{PDB_ID}.pdb', timeout=60).content
(OUT / f'{PDB_ID}.pdb').write_bytes(pdb_bytes)
print('sha256:', hashlib.sha256(pdb_bytes).hexdigest())


In [ ]:
entry = requests.get(f'https://data.rcsb.org/rest/v1/core/entry/{PDB_ID}', timeout=60).json()
chain_map = {}
for entity_id in entry['rcsb_entry_container_identifiers']['polymer_entity_ids']:
    entity = requests.get(f'https://data.rcsb.org/rest/v1/core/polymer_entity/{PDB_ID}/{entity_id}', timeout=60).json()
    desc = entity['rcsb_polymer_entity']['pdbx_description']
    for chain in entity['rcsb_polymer_entity_container_identifiers']['auth_asym_ids']: chain_map[chain] = desc
print(chain_map)
assert chain_map['A'].lower() == 'programmed cell death 1 ligand 1'
assert chain_map['B'].lower() == 'programmed cell death protein 1'


In [ ]:
def parse_atoms(path):
    atoms = []
    for line in path.read_text().splitlines():
        if not line.startswith('ATOM  '): continue
        atom = line[12:16].strip()
        element = (line[76:78].strip() or atom[0]).upper()
        if element == 'H' or atom.startswith('H'): continue
        chain = line[21:22].strip()
        if chain not in {'A','B'}: continue
        atoms.append({'chain':chain, 'residue':f'{line[17:20].strip()}{int(line[22:26])}{line[26:27].strip()}', 'x':float(line[30:38]), 'y':float(line[38:46]), 'z':float(line[46:54])})
    return atoms
atoms = parse_atoms(OUT / f'{PDB_ID}.pdb')
a_atoms = [a for a in atoms if a['chain'] == 'A']; b_atoms = [a for a in atoms if a['chain'] == 'B']
rows = {}
for a in a_atoms:
    for b in b_atoms:
        d = math.dist((a['x'],a['y'],a['z']), (b['x'],b['y'],b['z']))
        if d <= 4.5:
            row = rows.setdefault(a['residue'], {'min_distance_A':999.0, 'atom_contact_count':0, 'partner_residues':set()})
            row['min_distance_A'] = min(row['min_distance_A'], d); row['atom_contact_count'] += 1; row['partner_residues'].add(b['residue'])
with (OUT / 'interface_residues.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['residue','chain','role','source','min_distance_A','atom_contact_count','partner_residues','notes']); writer.writeheader()
    for residue, row in sorted(rows.items(), key=lambda item: int(''.join(c for c in item[0] if c.isdigit()))):
        writer.writerow({'residue':residue,'chain':'A','role':'PD-L1 interface contact','source':'PDB 4ZQK ATOM records; heavy-atom cutoff 4.5 A','min_distance_A':round(row['min_distance_A'],3),'atom_contact_count':row['atom_contact_count'],'partner_residues':';'.join(sorted(row['partner_residues'])),'notes':'Geometric contact; not automatically a functional hotspot'})
print('PD-L1 interface residues:', len(rows))


## Expected verified result for 4ZQK

With the current PDB ATOM records and a 4.5 Angstrom heavy-atom cutoff, chain A is PD-L1, chain B is PD-1, and 18 PD-L1 residues have cross-chain contacts. The checked-in CSV is the generated output.
